In [3]:
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import genpareto
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import gradio as gr
import matplotlib.pyplot as plt
import os
import zipfile
import glob

# ==========================================
# 1. Data Retrieval
# ==========================================
def download_era5_data():
    """Downloads multi-year ERA5 data by chunking requests year-by-year."""
    years = ['2020', '2021', '2022', '2023']
    c = cdsapi.Client()
    
    for year in years:
        filename = f'era5_multivariate_data_{year}.nc'
        if os.path.exists(filename):
            print(f"File {filename} already exists. Skipping.")
            continue
            
        print(f"Downloading ERA5 data for {year}...")
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'format': 'netcdf',
                'variable': [
                    '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind',
                    'total_cloud_cover', 'total_precipitation', '2m_dewpoint_temperature'
                ],
                'year': [year], # Requesting one year at a time
                'month': [str(i).zfill(2) for i in range(1, 13)],
                'day': [str(i).zfill(2) for i in range(1, 32)],
                'time': [f"{str(i).zfill(2)}:00" for i in range(0, 24, 6)],
                'area': [20.0, 72.0, 18.0, 74.0], 
            },
            filename)
    print("All yearly chunks downloaded successfully!")

# ==========================================
# 2. Dataset Processing
# ==========================================
def load_multi_location_era5(file_pattern='era5_multivariate_data_*.nc'):
    """Extracts zipped CDS downloads, normalizes coordinates, and builds a clean DataFrame."""
    extracted_files = []
    
    # 1. Extract files from disguised zip archives
    for disguised_zip_path in sorted(glob.glob(file_pattern)):
        if zipfile.is_zipfile(disguised_zip_path):
            extract_dir = f"extracted_{os.path.basename(disguised_zip_path).split('.')[0]}"
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(disguised_zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            extracted_files.extend(glob.glob(f"{extract_dir}/*.nc"))
        else:
            extracted_files.append(disguised_zip_path)
            
    if not extracted_files:
        raise ValueError("No internal NetCDF files could be extracted.")

    print(f"Loading {len(extracted_files)} NetCDF file(s) into DataFrames...")
    dfs = []
    for f in extracted_files:
        try:
            with xr.open_dataset(f, engine='netcdf4') as ds:
                # Harmonize time coordinate name: newer CDS uses 'valid_time', older uses 'time'
                if 'valid_time' in ds.coords or 'valid_time' in ds.dims:
                    ds = ds.rename({'valid_time': 'time'})
                
                # Convert this specific file to DataFrame
                df_sub = ds.to_dataframe().reset_index()
                dfs.append(df_sub)
        except Exception as e:
            print(f"Warning: Failed to read {f}: {e}")

    if not dfs:
        raise ValueError("Failed to load any data from the extracted NetCDF files.")

    # 2. Combine all tables
    print("Combining records...")
    df_raw = pd.concat(dfs, ignore_index=True)

    # 3. Rename variables dynamically to catch different CDS conventions
    col_mapping = {
        't2m': 'Temperature',
        '2t': 'Temperature',
        'd2m': 'Dewpoint',
        '2d': 'Dewpoint',
        'tcc': 'Cloud_Cover',
        'tp': 'Precipitation',
        'u10': 'u10',
        'v10': 'v10',
        '10u': 'u10',
        '10v': 'v10'
    }
    df_raw.rename(columns={k: v for k, v in col_mapping.items() if k in df_raw.columns}, inplace=True)

    # Calculate continuous wind speed
    if 'u10' in df_raw.columns and 'v10' in df_raw.columns:
        df_raw['Wind_Speed'] = np.sqrt(df_raw['u10']**2 + df_raw['v10']**2)
    elif 'Wind_Speed' not in df_raw.columns:
        df_raw['Wind_Speed'] = 0.0

    # Ensure latitude and longitude exist
    lat_col = 'latitude' if 'latitude' in df_raw.columns else 'lat'
    lon_col = 'longitude' if 'longitude' in df_raw.columns else 'lon'
    df_raw['location_id'] = df_raw[lat_col].round(2).astype(str) + '_' + df_raw[lon_col].round(2).astype(str)

    features = ['Temperature', 'Wind_Speed', 'Cloud_Cover', 'Precipitation']
    # Keep only available features
    available_features = [f for f in features if f in df_raw.columns]
    
    # 4. Merge instant and accumulated variables by grouping on (location_id, time)
    # Using 'first' or mean avoids losing rows where one variable is NaN in a specific sub-file
    print("Aggregating multi-variable timestamps...")
    df_clean = df_raw.groupby(['location_id', 'time'])[available_features].first().reset_index()
    
    # Forward-fill and backward-fill remaining missing values per location
    df_clean[available_features] = (
        df_clean.groupby('location_id')[available_features]
        .transform(lambda group: group.ffill().bfill())
    )
    
    df_clean = df_clean.dropna().sort_values(['location_id', 'time'])
    print(f"Loaded {len(df_clean)} rows across {df_clean['location_id'].nunique()} unique location(s).")
    return df_clean, available_features

def deseasonalize_and_scale(df, feature_cols):
    """Removes seasonality and scales data to [0, 1]."""
    df_deseasonalized = pd.DataFrame()
    scaler = MinMaxScaler()
    
    for col in feature_cols:
        trend = df[col].rolling(window=30, center=True).mean().bfill().ffill()
        df_deseasonalized[col] = df[col] - trend
        
    scaled_data = scaler.fit_transform(df_deseasonalized)
    return scaled_data, scaler

# ==========================================
# 3. Extreme Event Definition
# ==========================================
def identify_extremes_pot_gpd(data, threshold_percentile=95):
    """Uses POT and GPD to flag statistically justified extreme events."""
    threshold = np.percentile(data, threshold_percentile)
    exceedances = data[data > threshold] - threshold
    
    if len(exceedances) > 0:
        c, loc, scale = genpareto.fit(exceedances)
        gpd_threshold = genpareto.ppf(0.99, c, loc=loc, scale=scale) + threshold
    else:
        gpd_threshold = threshold
        
    is_extreme = data > gpd_threshold
    return is_extreme

def create_windowed_tensors(data, extremes, seq_length, pred_length):
    X, Y, Y_extremes = [], [], []
    for i in range(len(data) - seq_length - pred_length):
        X.append(data[i:(i + seq_length)])
        Y.append(data[(i + seq_length):(i + seq_length + pred_length), 0]) # Predicting Temp
        Y_extremes.append(extremes[(i + seq_length):(i + seq_length + pred_length)])
    return np.array(X), np.array(Y), np.array(Y_extremes)

# ==========================================
# 4. Forecasting Backbone
# ==========================================
class QuantileCNNLSTM(nn.Module):
    def __init__(self, input_size, seq_len, pred_len, quantiles):
        super(QuantileCNNLSTM, self).__init__()
        self.quantiles = quantiles
        self.pred_len = pred_len
        
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.lstm = nn.LSTM(input_size=64, hidden_size=50, batch_first=True, num_layers=2)
        self.fc = nn.Linear(50, pred_len * len(quantiles))

    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.cnn(x)
        x = x.permute(0, 2, 1) 
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out.view(-1, self.pred_len, len(self.quantiles))

def pinball_loss(y_true, y_pred, quantiles):
    loss = 0
    for i, q in enumerate(quantiles):
        error = y_true - y_pred[:, :, i]
        loss += torch.max((q - 1) * error, q * error).mean()
    return loss

# ==========================================
# 5. Evaluation Metrics
# ==========================================
def evaluate_performance(y_true, y_pred_lower, y_pred_median, y_pred_upper):
    if len(y_true) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "PICP": np.nan, "MPIW": np.nan}
        
    rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred_median.flatten()))
    mae = mean_absolute_error(y_true.flatten(), y_pred_median.flatten())
    
    covered = ((y_true >= y_pred_lower) & (y_true <= y_pred_upper)).astype(float)
    picp = np.mean(covered) 
    mpiw = np.mean(y_pred_upper - y_pred_lower) 
    
    return {"RMSE": rmse, "MAE": mae, "PICP": picp, "MPIW": mpiw}

# ==========================================
# 6. Gradio Interface
# ==========================================
def predict_interface(days_history):
    history = np.random.normal(0, 1, (1, int(days_history), 4)) # 4 features
    history_tensor = torch.tensor(history, dtype=torch.float32)
    
    model = QuantileCNNLSTM(input_size=4, seq_len=int(days_history), pred_len=12, quantiles=[0.05, 0.5, 0.95])
    model.eval()
    with torch.no_grad():
        preds = model(history_tensor).numpy()[0]
    
    lower_bound, median, upper_bound = preds[:, 0], preds[:, 1], preds[:, 2]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, 13), median, label='Median Forecast', color='blue', marker='o')
    ax.fill_between(range(1, 13), lower_bound, upper_bound, color='red', alpha=0.2, label='90% Confidence Interval')
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Forecast Horizon (Steps)")
    ax.set_ylabel("Standardized Output")
    ax.set_title("Forecast with Calibrated Uncertainty Estimates")
    ax.legend()
    
    return fig

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    FILENAME = 'era5_multivariate_data.nc'
    SEQ_LEN = 72
    PRED_LEN = 12
    QUANTILES = [0.05, 0.5, 0.95]
    EPOCHS = 5
    FEATURES = ['Temperature', 'Wind_Speed', 'Cloud_Cover', 'Precipitation']
    
    print("1. Initiating Data Download (Year-by-Year)...")
    download_era5_data() # No filename needed, handled in function
    
    print("2. Loading and Preprocessing ERA5 Data...")
    df_real, FEATURES = load_multi_location_era5('era5_multivariate_data_*.nc') 
    
    all_X, all_Y, all_Y_ext = [], [], []
    
    print("3. Processing independent spatial locations and calculating extremes...")
    X_train_list, Y_train_list = [], []
    X_test_list, Y_test_list, Y_ext_test_list = [], [], []

    for loc_id, group in df_real.groupby('location_id'):
        loc_data = group[FEATURES]
        
        if len(loc_data) <= (SEQ_LEN + PRED_LEN):
            print(f"Skipping location {loc_id}: sequence length ({len(loc_data)}) is too short.")
            continue
            
        scaled_data, scaler = deseasonalize_and_scale(loc_data, FEATURES)
        is_extreme = identify_extremes_pot_gpd(scaled_data[:, 0]) # Target: Temperature
        
        X_loc, Y_loc, Y_ext_loc = create_windowed_tensors(scaled_data, is_extreme, SEQ_LEN, PRED_LEN)
        
        if len(X_loc) > 0:
            # 80/20 chronological split for THIS specific location
            loc_split = int(0.8 * len(X_loc))
            
            X_train_list.append(X_loc[:loc_split])
            Y_train_list.append(Y_loc[:loc_split])
            
            X_test_list.append(X_loc[loc_split:])
            Y_test_list.append(Y_loc[loc_split:])
            Y_ext_test_list.append(Y_ext_loc[loc_split:])

    if not X_train_list:
        raise ValueError(
            f"No sequences could be created. Ensure each location has more than {SEQ_LEN + PRED_LEN} steps."
        )

    # Concatenate all training portions and testing portions across locations
    X_train_arr = np.concatenate(X_train_list, axis=0)
    Y_train_arr = np.concatenate(Y_train_list, axis=0)
    
    X_test_arr = np.concatenate(X_test_list, axis=0)
    Y_test = np.concatenate(Y_test_list, axis=0)
    Y_ext_test = np.concatenate(Y_ext_test_list, axis=0)

    print(f"Train shapes -> X: {X_train_arr.shape}, Y: {Y_train_arr.shape}")
    print(f"Test shapes  -> X: {X_test_arr.shape}, Y: {Y_test.shape}, Extremes: {Y_ext_test.shape}")

    # Convert to PyTorch tensors
    X_train = torch.tensor(X_train_arr, dtype=torch.float32)
    Y_train = torch.tensor(Y_train_arr, dtype=torch.float32)
    X_test = torch.tensor(X_test_arr, dtype=torch.float32)

    dataset = TensorDataset(X_train, Y_train)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)

    print("4. Training Quantile CNN-LSTM Architecture...")
    model = QuantileCNNLSTM(input_size=len(FEATURES), seq_len=SEQ_LEN, pred_len=PRED_LEN, quantiles=QUANTILES)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = pinball_loss(batch_y, preds, QUANTILES)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"   Epoch {epoch+1}/{EPOCHS} - Pinball Loss: {total_loss/len(loader):.4f}")
    
    print("5. Evaluating Normal vs. Extreme Subsets...")
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test).numpy()
    
    y_pred_lower, y_pred_median, y_pred_upper = test_preds[:, :, 0], test_preds[:, :, 1], test_preds[:, :, 2]
    
    normal_mask = ~Y_ext_test
    extreme_mask = Y_ext_test
    
    metrics_normal = evaluate_performance(Y_test[normal_mask], y_pred_lower[normal_mask], y_pred_median[normal_mask], y_pred_upper[normal_mask])
    metrics_extreme = evaluate_performance(Y_test[extreme_mask], y_pred_lower[extreme_mask], y_pred_median[extreme_mask], y_pred_upper[extreme_mask])
    
    print("\n--- PERFORMANCE & CALIBRATION METRICS ---")
    print(f"Normal Events  -> PICP: {metrics_normal['PICP']:.2f} | MPIW: {metrics_normal['MPIW']:.2f} | RMSE: {metrics_normal['RMSE']:.2f} | MAE: {metrics_normal['MAE']:.2f}")
    
    if np.isnan(metrics_extreme['RMSE']):
        print("Extreme Events -> No extreme events fell into the random test split.")
    else:
        print(f"Extreme Events -> PICP: {metrics_extreme['PICP']:.2f} | MPIW: {metrics_extreme['MPIW']:.2f} | RMSE: {metrics_extreme['RMSE']:.2f} | MAE: {metrics_extreme['MAE']:.2f}")

    print("\n6. Launching Web Interface...")
    iface = gr.Interface(
        fn=predict_interface,
        inputs=gr.Slider(minimum=24, maximum=120, step=12, label="Historical Input Window (Time Steps)"),
        outputs="plot",
        title="Multivariate Extreme Weather Forecaster"
    )
    iface.launch(share=False)

1. Initiating Data Download (Year-by-Year)...


2026-10-19 11:53:22,489 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**


File era5_multivariate_data_2020.nc already exists. Skipping.
File era5_multivariate_data_2021.nc already exists. Skipping.
File era5_multivariate_data_2022.nc already exists. Skipping.
File era5_multivariate_data_2023.nc already exists. Skipping.
All yearly chunks downloaded successfully!
2. Loading and Preprocessing ERA5 Data...
Loading 8 NetCDF file(s) into DataFrames...
Combining records...
Aggregating multi-variable timestamps...
Loaded 473364 rows across 81 unique location(s).
3. Processing independent spatial locations and calculating extremes...
Train shapes -> X: (373248, 72, 4), Y: (373248, 12)
Test shapes  -> X: (93312, 72, 4), Y: (93312, 12), Extremes: (93312, 12)
4. Training Quantile CNN-LSTM Architecture...
   Epoch 1/5 - Pinball Loss: 0.0475
   Epoch 2/5 - Pinball Loss: 0.0385
   Epoch 3/5 - Pinball Loss: 0.0354
   Epoch 4/5 - Pinball Loss: 0.0332
   Epoch 5/5 - Pinball Loss: 0.0318
5. Evaluating Normal vs. Extreme Subsets...

--- PERFORMANCE & CALIBRATION METRICS ---
No